# Lab W8: RunPod Pelatihan Jarak Jauh

Terkait: **Bab 08 · Adopsi Platform dan Tool Baru**

## Tujuan
1. Nyalakan pod RunPod (RTX 3090 spot instance), setup repo, jalankan 1 seed training penuh di PathMNIST.
2. Monitor via SSH tunnel TensorBoard.
3. Tarik checkpoint & log via rsync.
4. Matikan pod, verifikasi biaya total < $3.
5. Tulis `docs/tools/runpod.md` personal cheat sheet.

## Catatan
Notebook ini sebagian besar *log* aktivitas lokal (perintah SSH, rsync) - kode Python minimal. Jalankan perintah shell di terminal lokal, dokumentasikan di sel markdown.
Terkait: **Bab 08 · Adopsi Platform dan Tool Baru**

## Tujuan
1. Nyalakan pod RunPod (RTX 3090 spot instance), setup repo, jalankan 1 seed training penuh di PathMNIST.
2. Monitor via SSH tunnel TensorBoard.
3. Tarik checkpoint & log via rsync.
4. Matikan pod, verifikasi biaya total < $3.
5. Tulis `docs/tools/runpod.md` personal cheat sheet.

## Catatan
Notebook ini sebagian besar *log* aktivitas lokal (perintah SSH, rsync) - kode Python minimal. Jalankan perintah shell di terminal lokal, dokumentasikan di sel markdown.

## 1. Persiapan lokal

```bash
git status                     # clean
git log -1 --oneline
python -m src.train --config configs/baseline.yaml --dry-run    # harus lulus lokal
git push
```

## 2. Nyalakan pod

TODO: screenshot dashboard RunPod, catat:
- Tipe GPU: RTX 3090
- Disk ephemeral: 20 GB
- Network volume: (jika dipakai untuk dataset persisten)
- Estimasi biaya/jam: $...
- Waktu mulai: YYYY-MM-DD HH:MM

## 3. Setup pod

```bash
ssh -p <port> root@<host>
# di pod:
cd /workspace
git clone https://github.com/<akun>/<repo>.git
cd <repo>
pip install -e .
python -c "import torch; print(torch.cuda.get_device_name(0))"
python -m src.train --config configs/baseline.yaml --dry-run
```

## 4. Training penuh di tmux

```bash
tmux new -s train
python -m src.train --config configs/baseline.yaml --seed 42 \
  --output_dir /workspace/experiments 2>&1 | tee /workspace/experiments/baseline_seed42/train.log
# Ctrl+B, D untuk detach
```

## 5. Monitor: TensorBoard via SSH tunnel

Di terminal laptop terpisah:
```bash
ssh -L 6006:localhost:6006 -p <port> root@<host>
# di pod:
tensorboard --logdir /workspace/experiments --bind_all
```
Buka `http://localhost:6006` di browser.

## 6. Tarik hasil

```bash
rsync -avz --progress -e "ssh -p <port>" \
    root@<host>:/workspace/experiments/ ./experiments_from_pod/
```

## 7. Matikan pod + audit biaya

TODO: screenshot dashboard menunjukkan status `Stopped` dan biaya total. Catat di sini sebagai referensi ke future pod.

## 8. Cheat sheet pribadi (isi di `docs/tools/runpod.md`)

5 command paling sering + 3 gotcha pribadi yang kamu temukan. Bukan menyalin dari tutorial.